# Stage 1 - Gemma 4 QA Fine-tuning

This notebook prepares the Colab runtime, inspects the GPU environment, uploads a training zip, and fine-tunes a Gemma 4 model with checkpoint and resume support.

If you run this from VS Code with the Google Colab extension:
- connect the notebook to `Colab > Auto Connect`
- optionally use `Upload to Colab` on this repository if you want to run your local uncommitted files
- then execute the setup cells below

In [ ]:
#@title 1. Prepare repository on the Colab runtime
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Yongjooon/QA-AI-FineTunning.git"
REPO_NAME = "QA-AI-FineTunning"
REPO_DIR = Path(f"/content/{REPO_NAME}")

OVERRIDE_REPO_DIR = os.environ.get("QA_FINETUNE_REPO_DIR", "").strip()

if OVERRIDE_REPO_DIR and Path(OVERRIDE_REPO_DIR, "pyproject.toml").exists():
    REPO_DIR = Path(OVERRIDE_REPO_DIR)
    print(f"Using override repository directory: {REPO_DIR}")
else:
    if REPO_DIR.exists():
        print(f"Removing old repository: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)

    print(f"Cloning latest repository from {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.environ["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

os.chdir(REPO_DIR)
print("Current working directory:", os.getcwd())
print("Notebook file exists:", (REPO_DIR / "colab" / "01_train_gemma_qa.ipynb").exists())
print("Train script exists:", (REPO_DIR / "src" / "qafinetune" / "train.py").exists())
print("PYTHONPATH:", os.environ["PYTHONPATH"])


In [ ]:
#@title 2. Install dependencies
%cd /content/QA-AI-FineTunning
%pip install -q -U pip
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps

In [ ]:
#@title 3. Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 4. Optional Hugging Face login if the model is gated
# Gemma 4 is generally public on Hugging Face, but keep this cell for future gated models.
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
#@title 5. Training parameters
MODEL_NAME = "google/gemma-4-E2B-it"  # Gemma 4 model recommended for a T4 Colab runtime
PERSIST_ROOT = "/content/drive/MyDrive/qa_ai_finetuning"  # Persistent output location
RUN_NAME = "gemma4_qa_lora_run_01"
RESUME_MODE = "auto"  # auto | never | path
RESUME_CHECKPOINT = ""  # Only used when RESUME_MODE == 'path'
NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 2e-4
SAVE_STEPS = 50
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 4
MAX_TRAIN_SAMPLES = 0  # 0 keeps all records

In [ ]:
#@title 6. Inspect Colab runtime
import json
import sys
from pathlib import Path

repo_src = Path('/content/QA-AI-FineTunning/src')
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from qafinetune.runtime import detect_runtime, suggest_training_preset

runtime_profile = detect_runtime()
print(json.dumps(runtime_profile, indent=2, ensure_ascii=False))
print("\nSuggested preset:")
try:
    print(json.dumps(suggest_training_preset(runtime_profile), indent=2, ensure_ascii=False))
except RuntimeError as exc:
    print(f"Preset unavailable yet: {exc}")
    print("Switch the Colab runtime to GPU and rerun this cell before training.")

In [ ]:
#@title 7. Optional Smankusors Colab Monitor
# This executes a third-party remote script. It is disabled by default on purpose.
ENABLE_COLAB_MONITOR = False  #@param {type:"boolean"}

if ENABLE_COLAB_MONITOR:
    from urllib.request import urlopen
    exec(urlopen("http://colab-monitor.smankusors.com/track.py").read())
    _colabMonitor = ColabMonitor().start()
else:
    print("Colab Monitor is disabled.")

In [ ]:
#@title 8. Upload training zip
from google.colab import files
uploaded = files.upload()
TRAIN_ZIP_PATH = next(iter(uploaded.keys()))
print("Uploaded:", TRAIN_ZIP_PATH)

In [ ]:
#@title 9. Run training
import subprocess

command = [
    "python", "-m", "qafinetune.train",
    "--model_name", MODEL_NAME,
    "--train_zip", TRAIN_ZIP_PATH,
    "--output_root", PERSIST_ROOT,
    "--run_name", RUN_NAME,
    "--resume_mode", RESUME_MODE,
    "--resume_checkpoint", RESUME_CHECKPOINT,
    "--num_train_epochs", str(NUM_TRAIN_EPOCHS),
    "--learning_rate", str(LEARNING_RATE),
    "--save_steps", str(SAVE_STEPS),
    "--logging_steps", str(LOGGING_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--max_train_samples", str(MAX_TRAIN_SAMPLES),
]
print(" ".join(command))
subprocess.run(command, check=True)

In [ ]:
#@title 10. Inspect saved run state
import json
from pathlib import Path

run_state_path = Path(PERSIST_ROOT) / "train_runs" / RUN_NAME / "run_state.json"
print(run_state_path)
print(json.dumps(json.loads(run_state_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))